# Model Comparison and Selection

Compare all trained models across:
- Performance metrics (IC, Sharpe, hit rate)
- Statistical significance (MCPT p-values)
- Regime-conditional performance
- Feature importance consistency

Goal: Identify the best models for each target and universe.

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.ml import (
    ModelRegistry,
    TrainedModel,
    MODEL_CONFIGS,
    UNIVERSE_CONFIGS,
    TARGETS,
)

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load Trained Models

In [ ]:
# Initialize model registry
registry = ModelRegistry(Path("~/quant_results/models").expanduser())

# Load all models from disk
n_loaded = registry.load_all_from_disk()
print(f"Loaded {n_loaded} models")

# List all models
all_models = registry.list_models(validated_only=False)
validated_models = registry.list_models(validated_only=True)

print(f"\nTotal models: {len(all_models)}")
print(f"Validated models: {len(validated_models)}")

In [ ]:
# Build comparison table
comparison_data = []

for model_id in all_models:
    meta = registry.get_metadata(model_id)
    try:
        model = registry.load(model_id)
        comparison_data.append({
            'model_id': model_id,
            'model_type': meta.get('model_type', 'unknown'),
            'target': meta.get('target', 'unknown'),
            'is_validated': meta.get('is_validated', False),
            'mcpt_p_value': meta.get('mcpt_p_value', np.nan),
            'mean_test_score': model.metrics.get('mean_test_score', np.nan),
            'train_date': model.train_date,
        })
    except Exception as e:
        print(f"Error loading {model_id}: {e}")

comparison_df = pd.DataFrame(comparison_data)
comparison_df.head(10)

## 2. Performance by Target

In [ ]:
# Aggregate by target
if len(comparison_df) > 0:
    target_summary = comparison_df.groupby('target').agg({
        'model_id': 'count',
        'is_validated': 'sum',
        'mean_test_score': 'mean',
        'mcpt_p_value': 'mean',
    }).rename(columns={
        'model_id': 'n_models',
        'is_validated': 'n_validated',
    })
    
    print("Performance by Target:")
    display(target_summary)
else:
    print("No trained models found. Run the training pipeline first.")

In [ ]:
# Plot performance by target and model type
if len(comparison_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Test score by target
    ax = axes[0]
    comparison_df.boxplot(column='mean_test_score', by='target', ax=ax)
    ax.set_title('Test Score by Target')
    ax.set_xlabel('Target')
    ax.set_ylabel('Mean Test Score')
    plt.suptitle('')
    
    # P-value distribution
    ax = axes[1]
    comparison_df['mcpt_p_value'].hist(bins=20, ax=ax)
    ax.axvline(0.05, color='r', linestyle='--', label='p=0.05')
    ax.set_title('MCPT P-Value Distribution')
    ax.set_xlabel('P-Value')
    ax.legend()
    
    plt.tight_layout()
    plt.show()

## 3. Best Models per Target

In [ ]:
# Find best model for each target
if len(comparison_df) > 0:
    validated_df = comparison_df[comparison_df['is_validated']]
    
    if len(validated_df) > 0:
        best_models = validated_df.loc[
            validated_df.groupby('target')['mean_test_score'].idxmax()
        ]
        
        print("Best Validated Model per Target:")
        display(best_models[['model_id', 'target', 'model_type', 'mean_test_score', 'mcpt_p_value']])
    else:
        print("No validated models found.")

## 4. Model Type Comparison

In [ ]:
# Compare model types
if len(comparison_df) > 0:
    model_type_summary = comparison_df.groupby('model_type').agg({
        'model_id': 'count',
        'is_validated': 'mean',  # Validation rate
        'mean_test_score': ['mean', 'std'],
        'mcpt_p_value': 'mean',
    })
    model_type_summary.columns = ['n_models', 'validation_rate', 'mean_score', 'score_std', 'mean_p_value']
    
    print("Performance by Model Type:")
    display(model_type_summary.sort_values('mean_score', ascending=False))

In [ ]:
# Heatmap of model type vs target
if len(comparison_df) > 0:
    pivot = comparison_df.pivot_table(
        values='mean_test_score',
        index='model_type',
        columns='target',
        aggfunc='mean'
    )
    
    if pivot.size > 0:
        plt.figure(figsize=(10, 6))
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', center=0.5)
        plt.title('Mean Test Score: Model Type vs Target')
        plt.tight_layout()
        plt.show()

## 5. Statistical Significance Analysis

In [ ]:
# Analyze MCPT results
if len(comparison_df) > 0:
    significant_rate = comparison_df['is_validated'].mean()
    print(f"Overall significance rate: {significant_rate:.1%}")
    print(f"Models with p < 0.05: {comparison_df['is_validated'].sum()}")
    print(f"Models with p < 0.01: {(comparison_df['mcpt_p_value'] < 0.01).sum()}")

In [ ]:
# Scatter plot: Test Score vs P-Value
if len(comparison_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = ['green' if v else 'red' for v in comparison_df['is_validated']]
    
    ax.scatter(
        comparison_df['mean_test_score'],
        comparison_df['mcpt_p_value'],
        c=colors,
        alpha=0.7,
    )
    
    ax.axhline(0.05, color='red', linestyle='--', label='p=0.05 threshold')
    ax.axvline(0.5, color='gray', linestyle=':', alpha=0.5)
    
    ax.set_xlabel('Mean Test Score')
    ax.set_ylabel('MCPT P-Value')
    ax.set_title('Model Performance vs Statistical Significance')
    ax.legend()
    
    plt.tight_layout()
    plt.show()

## 6. Model Selection Recommendations

In [ ]:
# Generate recommendations
if len(comparison_df) > 0:
    recommendations = []
    
    for target in comparison_df['target'].unique():
        target_models = comparison_df[
            (comparison_df['target'] == target) &
            (comparison_df['is_validated'])
        ].sort_values('mean_test_score', ascending=False)
        
        if len(target_models) > 0:
            best = target_models.iloc[0]
            recommendations.append({
                'Target': target,
                'Best Model': best['model_type'],
                'Model ID': best['model_id'],
                'Test Score': f"{best['mean_test_score']:.3f}",
                'P-Value': f"{best['mcpt_p_value']:.4f}",
                'Status': 'Recommended' if best['mcpt_p_value'] < 0.01 else 'Acceptable',
            })
        else:
            recommendations.append({
                'Target': target,
                'Best Model': 'None',
                'Model ID': '-',
                'Test Score': '-',
                'P-Value': '-',
                'Status': 'Needs Training',
            })
    
    recommendations_df = pd.DataFrame(recommendations)
    print("\n=== Model Selection Recommendations ===")
    display(recommendations_df)

## 7. Summary Statistics

In [ ]:
# Final summary
if len(comparison_df) > 0:
    print("=== Model Comparison Summary ===")
    print(f"\nTotal models trained: {len(comparison_df)}")
    print(f"Validated models (p < 0.05): {comparison_df['is_validated'].sum()}")
    print(f"Validation rate: {comparison_df['is_validated'].mean():.1%}")
    print(f"\nBest overall test score: {comparison_df['mean_test_score'].max():.3f}")
    print(f"Lowest p-value: {comparison_df['mcpt_p_value'].min():.4f}")
    
    print(f"\nTargets with validated models:")
    for target in comparison_df[comparison_df['is_validated']]['target'].unique():
        n_models = len(comparison_df[(comparison_df['target'] == target) & comparison_df['is_validated']])
        print(f"  - {target}: {n_models} models")
else:
    print("No models to summarize. Train models using the ML training pipeline.")